In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


**Accuracy**  
Именно доля правильных ответов необходима для оценки обычного соревнования Spaceship Titanic

In [ ]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
    cross_val_score,
    cross_val_predict,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
CV_FOLDS = 5
FAST_CV_FOLDS = 3

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_fast = StratifiedKFold(
    n_splits=FAST_CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

sns.set_context("notebook")
pd.set_option("display.max_columns", 100)

## 1. Загрузка данных

Код ищет данные в нескольких стандартных местах. Это удобно, чтобы один и тот же ноутбук работал локально, в Colab и на Kaggle.

In [ ]:
candidate_dirs = [
    Path("/kaggle/input/competitions/spaceship-titanic"),
    Path("/content"),
    Path(".")
]

DATA_DIR = None
for folder in candidate_dirs:
    if (folder / "train.csv").exists() and (folder / "test.csv").exists():
        DATA_DIR = folder
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Не найдены train.csv и test.csv. "
        "Положите их рядом с ноутбуком или подключите Kaggle Dataset."
    )

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

sample_path = DATA_DIR / "sample_submission.csv"
sample_submission = pd.read_csv(sample_path) if sample_path.exists() else None

print("DATA_DIR:", DATA_DIR)
print("train:", train.shape)
print("test :", test.shape)
display(train.head())

In [ ]:
print("Колонки:")
print(train.columns.tolist())

print("\nТипы данных:")
display(train.dtypes.to_frame("dtype"))

print("\nДоля класса Transported:")
display(train["Transported"].value_counts(normalize=True).rename("share"))

## 2. EDA 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=train, x="Transported", ax=axes[0])
axes[0].set_title("Баланс целевого класса")
axes[0].set_xlabel("Transported")
axes[0].set_ylabel("Количество")

missing = train.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]

sns.barplot(x=missing.values * 100, y=missing.index, ax=axes[1])
axes[1].set_title("Пропуски в train, %")
axes[1].set_xlabel("% пропусков")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cryo_rate = train.groupby("CryoSleep", dropna=False)["Transported"].mean()
cryo_rate.plot(kind="bar", ax=axes[0])
axes[0].set_title("Доля Transported в зависимости от CryoSleep")
axes[0].set_ylabel("Transported rate")
axes[0].set_xlabel("CryoSleep")
axes[0].tick_params(axis="x", rotation=0)

planet_rate = (
    train.groupby("HomePlanet", dropna=False)["Transported"]
    .mean()
    .sort_values(ascending=False)
)
planet_rate.plot(kind="bar", ax=axes[1])
axes[1].set_title("Доля Transported по HomePlanet")
axes[1].set_ylabel("Transported rate")
axes[1].set_xlabel("HomePlanet")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### Что видно на EDA

- классы примерно сбалансированы `accuracy` будет адекватна
- пропуски есть почти во всех содержательных колонках 
- `CryoSleep`, происхождение пассажира, каюта и траты содержательны
- `PassengerId`, `Cabin`, `Name` стоит разделить и привязать к признакам

## 3. Feature Engineering

Создадим признаки, которые легко объяснить:

- `GroupSize` - размер группы из `PassengerId`
- `GroupMember` - номер пассажира внутри группы
- `IsSolo` - путешествует один
- `CabinDeck`, `CabinNum`, `CabinSide`, `CabinRegion` - разделение на части каюты
- `SurnameCount`  сколько людей с той же фамилией, вдруг найдеим семью с ~ одинаковым поведением
- `LogTotalSpend`  логарифм суммарных трат ( логарифм выступает нормировкой для значения потраченных денег)
- `NoSpend`  были ли траты вообще
- `AmenitiesUsed` сколько типов услуг использовал пассажир
- `AgeGroup`, `IsChild` простые возрастные группы



Размеры групп и количество фамилий считаем на объединении `train + test`, но без таргета

In [ ]:
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

all_features_only = pd.concat(
    [train.drop(columns="Transported"), test],
    axis=0,
    ignore_index=True
)
group_id_all = all_features_only["PassengerId"].str.split("_").str[0]
GROUP_SIZE_MAP = group_id_all.value_counts()

surname_all = (
    all_features_only["Name"]
    .fillna("Unknown Unknown")
    .str.split()
    .str[-1]
)
SURNAME_COUNT_MAP = surname_all.value_counts()

def make_features(df):
    df = df.copy()
    group_id = df["PassengerId"].str.split("_").str[0]
    df["GroupMember"] = pd.to_numeric(
        df["PassengerId"].str.split("_").str[1],
        errors="coerce"
    )
    df["GroupSize"] = group_id.map(GROUP_SIZE_MAP).astype(float)
    df["IsSolo"] = (df["GroupSize"] == 1).astype(int)
    cabin_parts = df["Cabin"].fillna("Unknown/-1/Unknown").str.split("/", expand=True)
    df["CabinDeck"] = cabin_parts[0]
    df["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    df["CabinSide"] = cabin_parts[2]
    df["CabinRegion"] = pd.cut(
        df["CabinNum"],
        bins=[-np.inf, 300, 600, 900, 1200, np.inf],
        labels=["0-300", "301-600", "601-900", "901-1200", "1201+"]
    ).astype("object")

    # фамилии
    surname = (
        df["Name"]
        .fillna("Unknown Unknown")
        .str.split()
        .str[-1]
    )
    df["SurnameCount"] = surname.map(SURNAME_COUNT_MAP).astype(float)

    # Траты
    spend = df[SPEND_COLS]
    total_spend_for_fe = spend.fillna(0).sum(axis=1)

    df["LogTotalSpend"] = np.log1p(total_spend_for_fe)
    df["NoSpend"] = (total_spend_for_fe == 0).astype(int)
    df["AmenitiesUsed"] = (spend.fillna(0) > 0).sum(axis=1)

    # Возраст
    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[-np.inf, 12, 18, 25, 40, 60, np.inf],
        labels=["child", "teen", "young", "adult", "mature", "senior"]
    ).astype("object")
    df["IsChild"] = (df["Age"] < 13).astype(int)
    df = df.drop(columns=["PassengerId", "Cabin", "Name"])

    return df


X = make_features(train.drop(columns="Transported"))
X_test = make_features(test)
y = train["Transported"].astype(int)

print("После feature engineering:", X.shape)
display(X.head())

## 4. Корреляции и линейная зависимость

Высокую корреляцию удалять не буду так как деревья с ней справляются, а у Logistic Regression будет регуляризация


In [ ]:
numeric_corr = X.select_dtypes(include=np.number).corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    numeric_corr,
    cmap="coolwarm",
    center=0,
    square=False,
    cbar_kws={"shrink": 0.8}
)
plt.title("Корреляции числовых признаков после Feature Engineering")
plt.tight_layout()
plt.show()

upper = numeric_corr.where(
    np.triu(np.ones(numeric_corr.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .rename("corr")
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)

high_corr_pairs["abs_corr"] = high_corr_pairs["corr"].abs()
high_corr_pairs = high_corr_pairs.query("abs_corr >= 0.85").sort_values(
    "abs_corr", ascending=False
)

print("Сильно коррелирующие пары (|corr| >= 0.85):")
display(high_corr_pairs if len(high_corr_pairs) else pd.DataFrame({"info": ["Таких пар нет"]}))

In [ ]:
def make_preprocessor(X_part):
    numeric_cols = X_part.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [c for c in X_part.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False
            )
        )
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])


def build_pipeline(model, X_part):
    return Pipeline([
        ("prep", make_preprocessor(X_part)),
        ("model", model)
    ])


diagnostic_prep = make_preprocessor(X)
X_diagnostic = diagnostic_prep.fit_transform(X)

matrix_rank = np.linalg.matrix_rank(X_diagnostic)
n_columns = X_diagnostic.shape[1]

print(f"Столбцов после preprocessing: {n_columns}")
print(f"Ранг матрицы:                {matrix_rank}")

if matrix_rank == n_columns:
    print("Точной линейной зависимости между итоговыми столбцами не обнаружено.")
else:
    print(
        "Есть точная линейная зависимость. "
        "Если вы добавляли свои признаки, проверьте признаки-суммы, дубли и dummy trap."
    )

Наиболее высокая корреляция наблюдается между LogTotalSpend и NoSpend. Однако признаки не являются линейно зависимыми: среди пассажиров с NoSpend = 0 величина LogTotalSpend принимает множество различных значений. Поэтому оба признака пока сохраняются, а их полезность далее проверяется с помощью feature importance и cross-validation

## 5. Baseline / Feature Engineering

Baseline 1: Logistic Regression на почти сырых данных.  
Baseline 2: LightGBM на почти сырых данных.  
Iteration: LightGBM после Feature Engineering.


In [ ]:
def cv_accuracy(name, pipeline, X_part, y_part, cv_object=cv):
    scores = cross_val_score(
        pipeline,
        X_part,
        y_part,
        cv=cv_object,
        scoring="accuracy",
        n_jobs=-1
    )
    result = {
        "experiment": name,
        "cv_mean": scores.mean(),
        "cv_std": scores.std()
    }
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")
    return result


raw_drop = ["Transported", "PassengerId", "Name", "Cabin"]
X_raw = train.drop(columns=raw_drop)

baseline_logreg = build_pipeline(
    LogisticRegression(
        max_iter=1500,
        C=1.0,
        solver="liblinear",
        random_state=RANDOM_STATE
    ),
    X_raw
)

baseline_lgbm = build_pipeline(
    LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=1
    ),
    X_raw
)

fe_lgbm = build_pipeline(
    LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=1
    ),
    X
)

experiment_log = []
experiment_log.append(cv_accuracy("Baseline Logistic (raw)", baseline_logreg, X_raw, y))
experiment_log.append(cv_accuracy("Baseline LightGBM (raw)", baseline_lgbm, X_raw, y))
experiment_log.append(cv_accuracy("LightGBM + Feature Engineering", fe_lgbm, X, y))

display(pd.DataFrame(experiment_log).sort_values("cv_mean", ascending=False))

По оценке видно, что Feature Engineering действительно дал прирост качества

Также LightGBM заметно превосходит базовую Logistic Regression: бустинг лучше улавливает нелинейные зависимости и взаимодействия между признаками.
При этом стандартное отклонение у модели с Feature Engineering небольшое, значит результат достаточно стабильный между фолдами. Поэтому на этом этапе можно сделать вывод, что создание новых признаков было полезным, и дальнейшие эксперименты стоит проводить именно на новом наборе признаков

## 6. Feature importance

Использую два вида:

1. `feature_importances_` - показывает важность уже после One-Hot Encoding.
2. Permutation importance - перемешивает один исходный признак и смотрит, насколько падает качество. Она удобнее для отбора, потому что сохраняет исходные названия колонок.


In [ ]:
X_train_imp, X_valid_imp, y_train_imp, y_valid_imp = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

importance_pipe = build_pipeline(
    LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=1
    ),
    X_train_imp
)

importance_pipe.fit(X_train_imp, y_train_imp)
valid_acc = accuracy_score(y_valid_imp, importance_pipe.predict(X_valid_imp))
print(f"Validation accuracy для модели важности: {valid_acc:.4f}")

prep = importance_pipe.named_steps["prep"]
lgbm_model = importance_pipe.named_steps["model"]

encoded_names = prep.get_feature_names_out()
built_in_importance = (
    pd.DataFrame({
        "feature": encoded_names,
        "importance": lgbm_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=built_in_importance.head(20),
    x="importance",
    y="feature"
)
plt.title("Top-20 LightGBM feature_importances_")
plt.xlabel("importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(built_in_importance.head(20))

In [ ]:
perm = permutation_importance(
    importance_pipe,
    X_valid_imp,
    y_valid_imp,
    scoring="accuracy",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=perm_importance.head(18),
    x="importance_mean",
    y="feature"
)
plt.title("Permutation importance исходных признаков")
plt.xlabel("Среднее падение accuracy при перемешивании")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(perm_importance)

## 7. Перебор количества признаков ( украла идею с семинара :) )

1. сортируем признаки по permutation importance
2. начинаем с нескольких лучших
3. добавляем по одному
4. каждый набор проверяем на CV
5. выбираем количество, где средняя `accuracy` максимальна


In [ ]:
feature_order = perm_importance["feature"].tolist()

min_features = min(5, len(feature_order))
feature_count_results = []

for k in range(min_features, len(feature_order) + 1):
    selected = feature_order[:k]

    model = build_pipeline(
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=1
        ),
        X[selected]
    )

    scores = cross_val_score(
        model,
        X[selected],
        y,
        cv=cv_fast,
        scoring="accuracy",
        n_jobs=-1
    )

    feature_count_results.append({
        "n_features": k,
        "cv_mean": scores.mean(),
        "cv_std": scores.std()
    })

feature_count_df = pd.DataFrame(feature_count_results)

best_row = feature_count_df.loc[feature_count_df["cv_mean"].idxmax()]
BEST_K = int(best_row["n_features"])
selected_features = feature_order[:BEST_K]

print(f"Лучшее число признаков: {BEST_K}")
print(f"Лучший 3-fold CV accuracy: {best_row['cv_mean']:.4f}")
print("\nОтобранные признаки:")
print(selected_features)

plt.figure(figsize=(10, 5))
plt.plot(
    feature_count_df["n_features"],
    feature_count_df["cv_mean"],
    marker="o"
)
plt.axvline(BEST_K, linestyle="--", label=f"best k = {BEST_K}")
plt.xlabel("Количество лучших признаков")
plt.ylabel("CV accuracy")
plt.title("Что происходит, когда добавляем признаки по одному")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

display(
    feature_count_df.sort_values("cv_mean", ascending=False).head(10)
)

итог: 16 признаков

In [ ]:
X_sel = X[selected_features].copy()
X_test_sel = X_test[selected_features].copy()

print("Финальный набор признаков:", X_sel.shape)
display(X_sel.head())

## 8. модели

Сравним:
- Logistic Regression
- LightGBM
- XGBoost
- CatBoost


In [ ]:
base_models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1500,
        C=1.0,
        solver="liblinear",
        random_state=RANDOM_STATE
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=350,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist"
    ),

    "CatBoost": CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
        thread_count=1
    )
}

base_model_results = []

for name, model in base_models.items():
    pipe = build_pipeline(model, X_sel)

    scores = cross_val_score(
        pipe,
        X_sel,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    base_model_results.append({
        "model": name,
        "cv_mean": scores.mean(),
        "cv_std": scores.std()
    })

base_model_results = pd.DataFrame(base_model_results).sort_values(
    "cv_mean", ascending=False
)

display(base_model_results)

plt.figure(figsize=(8, 4))
sns.barplot(data=base_model_results, x="cv_mean", y="model")
plt.xlim(
    max(0.5, base_model_results["cv_mean"].min() - 0.03),
    min(1.0, base_model_results["cv_mean"].max() + 0.03)
)
plt.title("4 обязательные модели: 5-fold CV accuracy")
plt.xlabel("CV accuracy")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 9. GridSearchCV

- `C` для Logistic Regression (сила регуляризации)
- `num_leaves`, `max_depth`, `min_child_samples` для LightGBM сложность дерева
- `max_depth`, `learning_rate`, `n_estimators` для XGBoost (глубина и темп бустинга)
- `depth`, `iterations`, `learning_rate` для CatBoost 

In [ ]:
search_spaces = {
    "LogisticRegression": {
        "model": LogisticRegression(
            max_iter=1500,
            solver="liblinear",
            random_state=RANDOM_STATE
        ),
        "params": {
            "model__C": [0.1, 0.3, 1.0, 3.0]
        }
    },

    "LightGBM": {
        "model": LGBMClassifier(
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [300, 500],
            "model__learning_rate": [0.03, 0.06],
            "model__num_leaves": [15, 31],
            "model__min_child_samples": [15, 30]
        }
    },

    "XGBoost": {
        "model": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            subsample=0.9,
            colsample_bytree=0.9
        ),
        "params": {
            "model__n_estimators": [300, 500],
            "model__max_depth": [4, 6],
            "model__learning_rate": [0.03, 0.06]
        }
    },

    "CatBoost": {
        "model": CatBoostClassifier(
            loss_function="Logloss",
            random_seed=RANDOM_STATE,
            verbose=0,
            allow_writing_files=False,
            thread_count=1
        ),
        "params": {
            "model__iterations": [400, 700],
            "model__depth": [5, 7],
            "model__learning_rate": [0.03, 0.06]
        }
    }
}

In [ ]:
best_models = {}
tuning_rows = []

for name, config in search_spaces.items():
    print("\n" + "=" * 70)
    print("Tuning:", name)

    pipe = build_pipeline(config["model"], X_sel)

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=config["params"],
        scoring="accuracy",
        cv=cv_fast,
        n_jobs=-1,
        refit=True,
        verbose=0
    )

    grid.fit(X_sel, y)

    best_models[name] = grid.best_estimator_

    tuning_rows.append({
        "model": name,
        "best_cv_accuracy": grid.best_score_,
        "best_params": grid.best_params_
    })

    print("Best CV accuracy:", round(grid.best_score_, 5))
    print("Best params:", grid.best_params_)

tuning_summary = pd.DataFrame(tuning_rows).sort_values(
    "best_cv_accuracy",
    ascending=False
)

display(tuning_summary)

## 10. сравнение tuned-моделей через OOF

In [ ]:
oof_probabilities = {}
oof_scores = []

for name, model in best_models.items():
    print("OOF:", name)

    proba = cross_val_predict(
        model,
        X_sel,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    oof_probabilities[name] = proba

    acc = accuracy_score(y, proba >= 0.5)
    oof_scores.append({
        "model": name,
        "oof_accuracy": acc
    })

oof_df = pd.DataFrame(oof_probabilities)
oof_score_df = pd.DataFrame(oof_scores).sort_values(
    "oof_accuracy",
    ascending=False
)

display(oof_score_df)

plt.figure(figsize=(7, 5))
sns.heatmap(oof_df.corr(), annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Корреляция OOF-вероятностей моделей")
plt.tight_layout()
plt.show()

## 11. Weighted Blending

In [ ]:
top3_models = oof_score_df.head(3)["model"].tolist()
print("Модели для blending:", top3_models)

weight_grid = np.arange(0, 1.001, 0.05)
threshold_grid = np.arange(0.46, 0.551, 0.01)

best_blend = {
    "accuracy": -1,
    "weights": None,
    "threshold": 0.5
}

p1 = oof_df[top3_models[0]].values
p2 = oof_df[top3_models[1]].values
p3 = oof_df[top3_models[2]].values

for w1 in weight_grid:
    for w2 in weight_grid:
        w3 = 1.0 - w1 - w2

        if w3 < -1e-9 or w3 > 1:
            continue

        blend_proba = w1 * p1 + w2 * p2 + w3 * p3

        for threshold in threshold_grid:
            acc = accuracy_score(y, blend_proba >= threshold)

            if acc > best_blend["accuracy"]:
                best_blend = {
                    "accuracy": acc,
                    "weights": [w1, w2, w3],
                    "threshold": float(threshold)
                }

print("Best blend OOF accuracy:", round(best_blend["accuracy"], 5))
print("Threshold:", best_blend["threshold"])

for model_name, weight in zip(top3_models, best_blend["weights"]):
    print(f"{model_name:20s}: {weight:.2f}")

## 12. Stacking

1. Первый уровень: 4 tuned-модели 
2. Второй уровень: Logistic Regression 


In [ ]:
meta_X = oof_df.copy()

meta_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="liblinear",
    random_state=RANDOM_STATE
)

stack_oof_proba = cross_val_predict(
    meta_model,
    meta_X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

stack_oof_accuracy = accuracy_score(y, stack_oof_proba >= 0.5)

print(f"Stacking OOF accuracy: {stack_oof_accuracy:.5f}")

meta_model.fit(meta_X, y)

stack_coefficients = pd.DataFrame({
    "base_model": meta_X.columns,
    "meta_coef": meta_model.coef_[0]
}).sort_values("meta_coef", ascending=False)

display(stack_coefficients)

## 13. Финальное обучение 

In [ ]:
test_probabilities = {}

for name, model in best_models.items():
    test_probabilities[name] = model.predict_proba(X_test_sel)[:, 1]

test_proba_df = pd.DataFrame(test_probabilities)

#Blend
blend_test_proba = np.zeros(len(test))

for model_name, weight in zip(top3_models, best_blend["weights"]):
    blend_test_proba += weight * test_proba_df[model_name].values

blend_pred = blend_test_proba >= best_blend["threshold"]
#Stack
stack_test_proba = meta_model.predict_proba(test_proba_df[oof_df.columns])[:, 1]
stack_pred = stack_test_proba >= 0.5

submission_blend = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": blend_pred.astype(bool)
})

submission_stacking = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": stack_pred.astype(bool)
})

submission_blend.to_csv("submission_blend.csv", index=False)
submission_stacking.to_csv("submission_stacking.csv", index=False)


print("submission_blend.csv")
print("submission_stacking.csv")

display(submission_blend.head())

In [ ]:
blend_oof_accuracy = best_blend["accuracy"]

if blend_oof_accuracy >= stack_oof_accuracy:
    final_method = "blend"
    submission = submission_blend.copy()
    final_oof = blend_oof_accuracy
else:
    final_method = "stacking"
    submission = submission_stacking.copy()
    final_oof = stack_oof_accuracy

submission.to_csv("submission.csv", index=False)

print(f"Рекомендованный final по OOF: {final_method}")
print(f"OOF accuracy: {final_oof:.5f}")
print("Файл: submission.csv")

display(submission.head())